# Notebook 1 — Data preparation: from the raw extraction to data04

Run from the `notebooks/` folder. Inputs come from `../data/raw/`; everything this notebook makes is written to `../outputs/`
(`data/`, `tables/`, `figures/`). Figures are PNG at 600 dpi in Cambria with the house colour scheme of `style.py`; tables are Excel in Cambria.
Run the notebooks in order: **1 → 2 → 3**.

| Part | Content | Writes |
|---|---|---|
| A | Raw manual extraction and YOLO crossing logs → timestamps, headways, speeds, flow; quality checks; data01 (898 events), data02 (894), data03 | `outputs/data/data01–03.xlsx` |
| B | Pooling tests for target and leader classes → data04 (894 events, 14 variables, 5 leader classes) | `outputs/data/data04.xlsx`, `outputs/tables/Pooling_Tables.xlsx`, three pooling figures |
| C | Appendix figures and tables: traffic state, event timing, detector performance | `outputs/figures/`, `outputs/tables/Appendix_Figure_Data.xlsx` |

The three camera-view stills of the sites are taken from the raw videos and are not regenerated here.

## Part A — Build datasets `data01` and `data02`

Rebuilds the analysis data from the two primary sources:

1. **Manual extraction**: `data/raw/Data Compiled (extracted observations).xlsx`, sheet `data`. This holds the four timestamps per following event, the vehicle labels and the behaviour flags.
2. **YOLO crossing logs**: `data/raw/yolo_crossings/*.xlsx`. Each row is a vehicle crossing the count line, with its class and time.

Outputs (in `data/`):
- `data01.xlsx`: all 898 events, with the main variables recomputed from the raw timestamps and the flow and composition recomputed from the YOLO logs.
- `data02.xlsx`: the cleaned analysis set. It drops the 4 events where the target reaches line 1 before the leader, leaving n = 894.

All paths are relative to this notebook, and every rule used is recorded in the `PARAMS` cell and in `LOG.md`.

In [ ]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path.cwd()))
from style import PAL, use_style
from pathlib import Path
import hashlib, json, datetime
import numpy as np
import pandas as pd

from openpyxl import load_workbook
from openpyxl.styles import Font
FONT_NAME = "Cambria"            # house font for all figures and Excel outputs
def cambria_xlsx(path):
    # set every cell of every sheet in an .xlsx to Cambria (header row bold)
    wb = load_workbook(path)
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for c in row:
                c.font = Font(name=FONT_NAME, bold=(c.row == 1), size=11)
    wb.save(path)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
OUT = ROOT / "outputs" / "data"; OUT.mkdir(parents=True, exist_ok=True)
print("project root:", ROOT)

### PARAMS: every rule used in this notebook

In [ ]:
FPS = 30                      # video frame rate used to convert frame -> seconds (all three sessions are 30 fps)
WINDOW_SEC = 30.0             # rolling flow window width (centred), seconds
EDGE_FRACTION = 0.8           # window flagged as edge-truncated if effective width < 0.8 * WINDOW_SEC
FLOW_TIME_RULE = "floor_TT2"  # flow evaluated at floor(TT2), the whole second in which the target crosses line 2
                              # (this reproduces the prior compiled dataset exactly; see check below)

# Passenger-car equivalents per YOLO class (RHD 2005, as used in the prior YOLO flow notebook)
PCU = {"3W": 2.00, "4W": 1.00, "Bicycle": 0.50, "Bike": 0.75, "Bus": 3.00, "CNG": 0.75}

# Session (video) -> YOLO crossing log, site, segment length (m), road width (m)
SESSIONS = {
    "SP":       {"log": "SP_log_M.xlsx",        "site": "Shahjahanpur", "L": 4.5, "W": 9.400},
    "SP2":      {"log": "SP2_log.xlsx",         "site": "Shahjahanpur", "L": 4.2, "W": 9.400},
    "Tikatuli": {"log": "Tikatuli Merged.xlsx", "site": "Tikatuli",     "L": 8.3, "W": 13.885},
}

# Target class: Mod_BTW is pooled into BTW (pooling to be re-tested in the analysis notebooks)
TARGET_MAP = {"BTW": "BTW", "Mod_BTW": "BTW", "PR": "PR"}

# Leader class: raw label -> five engineering classes (same scheme as the TRB data3 file)
LEADER_MAP = {"Car": "4W", "SUV": "4W", "Utility_V": "4W", "Man_Hauler": "4W",
              "BTW": "MT_3W", "Mod_BTW": "MT_3W", "CNG": "MT_3W",
              "PR": "NMT_3W", "Van": "NMT_3W",
              "Bike": "MT_2W", "Bicycle": "NMT_2W"}

# data02 cleaning rule
DROP_TARGET_AHEAD_AT_L1 = True   # drop events with TT1 < LT1

### 1. Input fingerprints

In [ ]:
def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

inputs = [RAW / "Data Compiled (extracted observations).xlsx"] + \
         [RAW / "yolo_crossings" / s["log"] for s in SESSIONS.values()]
fingerprints = pd.DataFrame({"file": [str(p.relative_to(ROOT)) for p in inputs],
                             "sha256": [sha256(p) for p in inputs],
                             "bytes": [p.stat().st_size for p in inputs]})
fingerprints

### 2. Manual extraction: load and rebuild timestamps

In [ ]:
raw = pd.read_excel(RAW / "Data Compiled (extracted observations).xlsx", sheet_name="data", header=1)
raw.insert(0, "Row_Source", np.arange(3, len(raw) + 3))   # Excel row number in the source sheet (header is on row 2)
print(raw.shape)

def ts(m, s, f):
    return raw[m] * 60 + raw[s] + raw[f] / FPS

t = pd.DataFrame({
    "LT1": ts("Min",  "Sec",  "Frame"),     # leader crosses line 1
    "TT1": ts("Min2", "Sec3", "Frame4"),    # target crosses line 1
    "LT2": ts("Min5", "Sec6", "Frame7"),    # leader crosses line 2
    "TT2": ts("Min8", "Sec9", "Frame10"),   # target crosses line 2
})

# Check against the timestamps stored in the source sheet
chk = {"LT1": "LT1", "TT1": "TT1", "LT2": "LT2", "TT2": "Video_Time_Stamp"}
for k, col in chk.items():
    err = (t[k] - raw[col]).abs().max()
    print(f"{k}: max |recomputed - stored| = {err:.2e}")
    assert err < 1e-6

### 3. Main variables, recomputed from timestamps

In [ ]:
d = pd.DataFrame({"Row_Source": raw["Row_Source"], "Session": raw["Video"]})
d["Site"] = d["Session"].map({k: v["site"] for k, v in SESSIONS.items()})
d["V_Target_Raw"] = raw["V_Target"]
d["Target_Class"] = raw["V_Target"].map(TARGET_MAP)
d["V_Leading_Raw"] = raw["V_Leading"]
d["Leader_Class"] = raw["V_Leading"].map(LEADER_MAP)
assert d[["Target_Class", "Leader_Class"]].notna().all().all(), "unmapped label"
d["Pair"] = d["Target_Class"] + "_following_" + d["Leader_Class"]

d[["LT1", "TT1", "LT2", "TT2"]] = t.values
d["Segment_Length_m"] = d["Session"].map({k: v["L"] for k, v in SESSIONS.items()})
assert np.allclose(d["Segment_Length_m"], raw["Segment_Length"])

d["Time_Headway_s"]        = d["TT2"] - d["LT2"]             # headway at line 2 (front-to-front passage)
d["Headway_L1_s"]          = d["TT1"] - d["LT1"]             # headway at line 1 (kept for reference/QA)
d["Target_Travel_Time_s"]  = d["TT2"] - d["TT1"]
d["Leader_Travel_Time_s"]  = d["LT2"] - d["LT1"]
d["Target_Speed_kmh"]      = d["Segment_Length_m"] / d["Target_Travel_Time_s"] * 3.6
d["Leading_Speed_kmh"]     = d["Segment_Length_m"] / d["Leader_Travel_Time_s"] * 3.6
d["Speed_Difference_kmh"]  = d["Target_Speed_kmh"] - d["Leading_Speed_kmh"]

# Manually coded behaviour / site variables (carried over unchanged)
d["Off_Centered"]        = raw["Off_centeredness"].astype(bool)
d["Occupied"]            = raw["Occupancy"].astype(bool)
d["Roadside_Parking"]    = raw["Road_Side_Parking"].astype(bool)
d["Road_Width_m"]        = d["Session"].map({k: v["W"] for k, v in SESSIONS.items()})
d["Parking_Width_m"]     = raw["Space_Consumed_by_Parking"]
d["Effective_Width_m"]   = d["Road_Width_m"] - d["Parking_Width_m"]

# Checks against the source sheet's derived columns
pairs = [("Time_Headway_s", "Time_Headway"), ("Target_Travel_Time_s", "Target_Gap"),
         ("Leader_Travel_Time_s", "Leading_Gap"), ("Target_Speed_kmh", "Target_Speed_km/hr"),
         ("Leading_Speed_kmh", "Leading_Speed_km/hr"), ("Speed_Difference_kmh", "Speed_Difference"),
         ("Road_Width_m", "Road_Width"), ("Effective_Width_m", "Available_Carriage_Way")]
for a, b in pairs:
    err = (d[a] - raw[b]).abs().max()
    print(f"{a:24s} vs {b:22s} max err = {err:.2e}")
    assert err < 1e-6

### 4. Rolling flow and composition from the YOLO crossing logs

For each session, the flow at time *t* is the PCU sum of the crossings in the centred window
[t − 15, t + 15] s, scaled to an hourly rate:

$$q(t) = \frac{3600}{w_{\text{eff}}(t)} \sum_{j:\,t_j\in[t-15,\,t+15]} \text{PCU}_{c_j}$$

The window is truncated at the start and end of the recording, so $w_{\text{eff}}\le 30$ s; windows narrower than 24 s are flagged as edge windows.
Composition shares are PCU fractions, $\text{share}_k = \text{PCU}_k / \text{PCU}_{\text{total}}$, all taken from the same window.
Flow per metre is $q / W_{\text{eff}}$, where $W_{\text{eff}}$ is the road width minus any parking encroachment.

In [ ]:
CLASSES = list(PCU)

def rolling_flow(log, tq):
    log = log.sort_values("time_sec")
    tt = log["time_sec"].to_numpy(float)
    cl = log["class_name"].to_numpy()
    unknown = set(cl) - set(PCU)
    assert not unknown, unknown
    t0, t1 = tt.min(), tt.max()
    lo = np.clip(tq - WINDOW_SEC / 2, t0, None)
    hi = np.clip(tq + WINDOW_SEC / 2, None, t1)
    eff = hi - lo
    il = np.searchsorted(tt, lo, "left")
    ih = np.searchsorted(tt, hi, "right")
    out = {"Flow_Window_Eff_s": eff, "Flow_Edge_Window": eff < EDGE_FRACTION * WINDOW_SEC}
    pcu = np.array([PCU[c] for c in cl])
    cum_pcu = np.r_[0, np.cumsum(pcu)]
    cum_veh = np.r_[0, np.arange(1, len(tt) + 1)]
    out["Flow_pcu_h"] = (cum_pcu[ih] - cum_pcu[il]) * 3600 / eff
    out["Flow_veh_h"] = (cum_veh[ih] - cum_veh[il]) * 3600 / eff
    tot = cum_pcu[ih] - cum_pcu[il]
    for k in CLASSES:
        ck = np.r_[0, np.cumsum(np.where(cl == k, pcu, 0.0))]
        out[f"Share_{k}"] = np.divide(ck[ih] - ck[il], tot, out=np.zeros_like(tot), where=tot > 0)
    return pd.DataFrame(out)

logs, flow_parts = {}, []
for s, cfg in SESSIONS.items():
    logs[s] = pd.read_excel(RAW / "yolo_crossings" / cfg["log"])
    idx = d.index[d["Session"] == s]
    tq = np.floor(d.loc[idx, "TT2"].to_numpy()) if FLOW_TIME_RULE == "floor_TT2" else d.loc[idx, "TT2"].to_numpy()
    f = rolling_flow(logs[s], tq)
    f.index = idx
    flow_parts.append(f)
    L = logs[s]
    print(f"{s:9s} crossings={len(L):5d}  span={L.time_sec.min():7.1f}-{L.time_sec.max():7.1f} s  "
          f"events={len(idx)}  event TT2 span={d.loc[idx,'TT2'].min():.1f}-{d.loc[idx,'TT2'].max():.1f} s")
flow = pd.concat(flow_parts).sort_index()
d = d.join(flow)
d["Share_3W_All"] = d["Share_3W"] + d["Share_CNG"]          # all three-wheelers (YOLO 3W + CNG)
d["Share_2W"] = d["Share_Bike"] + d["Share_Bicycle"]
d["Flow_pcu_h_m"] = d["Flow_pcu_h"] / d["Effective_Width_m"]

#### 4.1 Reproduction check against the prior compiled flow and shares

In [ ]:
repro = [("Flow_pcu_h", "Flow_pcu/hr"), ("Flow_pcu_h_m", "Flow/m"), ("Share_4W", "Share_4W"),
         ("Share_Bus", "Share_Bus"), ("Share_CNG", "Share_CNG"), ("Share_Bike", "Share_Bike"),
         ("Share_Bicycle", "Share_Bicycle"), ("Share_3W", "Share_3W_Except_CNG"),
         ("Share_3W_All", "Share_3W"), ("Share_2W", "Share_2W")]
rep = pd.DataFrame([{"new": a, "prior": b,
                     "n_match": int(np.isclose(d[a], raw[b], rtol=1e-6, atol=1e-9).sum()),
                     "max_abs_diff": float((d[a] - raw[b]).abs().max())} for a, b in repro])
rep["n"] = len(d)
print(rep.to_string(index=False))
assert (rep["n_match"] == len(d)).all(), "flow/share reproduction failed"

### 5. QA flags (kept in data01, used for the data02 rule)

In [ ]:
frames = raw[["Frame", "Frame4", "Frame7", "Frame10"]]
d["QA_Frame_GE_FPS"]         = (frames >= FPS).any(axis=1)          # retained: valid entry (see LOG)
d["QA_Target_Ahead_At_L1"]   = d["TT1"] < d["LT1"]                  # dropped in data02
d["QA_Shared_Leader"]        = d.duplicated(["Session", "LT1", "LT2"], keep=False)  # informational only
qa = d[["QA_Frame_GE_FPS", "QA_Target_Ahead_At_L1", "QA_Shared_Leader", "Flow_Edge_Window"]].sum()
print(qa)
d.loc[d["QA_Target_Ahead_At_L1"], ["Row_Source", "Session", "Pair", "LT1", "TT1", "LT2", "TT2", "Time_Headway_s", "Headway_L1_s"]]

### 6. Cross-check with the prior TRB analysis file `data3.xlsx` (leader classes and pairs)

In [ ]:
prior3 = RAW / "prior_trb" / "data3.xlsx"
if prior3.exists():
    p3 = pd.read_excel(prior3, sheet_name="Sheet1")
    # match rows on (headway, target speed, leader speed); compare pair labels as multisets per key
    key = lambda df, cols: df[cols].round(6).astype(str).agg("|".join, axis=1)
    ka = key(d, ["Time_Headway_s", "Target_Speed_kmh", "Leading_Speed_kmh"])
    kb = key(p3, ["Time_Headway", "Target_Speed_km/hr", "Leading_Speed_km/hr"])
    A = d.groupby(ka)["Pair"].apply(lambda s: tuple(sorted(s)))
    B = p3.groupby(kb)["Pair"].apply(lambda s: tuple(sorted(s)))
    common = A.index.intersection(B.index)
    agree = (A[common] == B[common]).mean()
    print(f"keys: data01={len(A)} data3={len(B)} common={len(common)}  pair-label agreement on common keys: {agree:.4f}")
    assert agree == 1.0
    print(pd.concat([d["Pair"].value_counts().rename("data01"), p3["Pair"].value_counts().rename("data3")], axis=1))
else:
    print("data3.xlsx not found; skipped (optional check)")

### 7. Assemble data01 and data02

In [ ]:
d.insert(0, "Obs_ID", np.arange(1, len(d) + 1))

COLS = ["Obs_ID", "Row_Source", "Session", "Site",
        "V_Target_Raw", "Target_Class", "V_Leading_Raw", "Leader_Class", "Pair",
        "LT1", "TT1", "LT2", "TT2", "Segment_Length_m",
        "Time_Headway_s", "Headway_L1_s", "Target_Travel_Time_s", "Leader_Travel_Time_s",
        "Target_Speed_kmh", "Leading_Speed_kmh", "Speed_Difference_kmh",
        "Off_Centered", "Occupied", "Roadside_Parking",
        "Road_Width_m", "Parking_Width_m", "Effective_Width_m",
        "Flow_pcu_h", "Flow_veh_h", "Flow_pcu_h_m", "Flow_Window_Eff_s", "Flow_Edge_Window",
        "Share_3W", "Share_CNG", "Share_3W_All", "Share_4W", "Share_Bus", "Share_Bike", "Share_Bicycle", "Share_2W",
        "QA_Frame_GE_FPS", "QA_Target_Ahead_At_L1", "QA_Shared_Leader"]
data01 = d[COLS].copy()

mask_drop = data01["QA_Target_Ahead_At_L1"] if DROP_TARGET_AHEAD_AT_L1 else pd.Series(False, index=data01.index)
dropped = data01.loc[mask_drop, ["Obs_ID", "Row_Source", "Session", "Pair", "Time_Headway_s"]].assign(
    Reason="Target crossed line 1 before leader (TT1 < LT1)")
data02 = data01.loc[~mask_drop].drop(columns=["QA_Target_Ahead_At_L1"]).reset_index(drop=True)
print("data01:", data01.shape, " data02:", data02.shape)

### 8. Variable dictionary

In [ ]:
DESC = {
 "Obs_ID": "Sequential ID in source order (stable across data01/data02)",
 "Row_Source": "Excel row in source sheet 'data' of Data Compiled (extracted observations).xlsx",
 "Session": "Video session: SP, SP2 (Shahjahanpur), Tikatuli (two halves merged on one timeline)",
 "Site": "Study corridor",
 "V_Target_Raw": "Target (follower) label as coded: PR, BTW, Mod_BTW",
 "Target_Class": "PR or BTW (Mod_BTW pooled into BTW)",
 "V_Leading_Raw": "Leader label as coded (11 classes)",
 "Leader_Class": "4W (Car, SUV, Utility_V, Man_Hauler); MT_3W (BTW, Mod_BTW, CNG); NMT_3W (PR, Van); MT_2W (Bike); NMT_2W (Bicycle)",
 "Pair": "Target_Class + '_following_' + Leader_Class",
 "LT1": "Leader passes reference line 1 [s, session timeline] = min*60 + sec + frame/30",
 "TT1": "Target passes reference line 1 [s]",
 "LT2": "Leader passes reference line 2 [s]",
 "TT2": "Target passes reference line 2 [s]",
 "Segment_Length_m": "Longitudinal distance between reference lines [m]",
 "Time_Headway_s": "TT2 - LT2: time headway at line 2 [s] (analysis outcome)",
 "Headway_L1_s": "TT1 - LT1: time headway at line 1 [s] (reference/QA only)",
 "Target_Travel_Time_s": "TT2 - TT1 [s] (named Target_Gap in prior file)",
 "Leader_Travel_Time_s": "LT2 - LT1 [s] (named Leading_Gap in prior file)",
 "Target_Speed_kmh": "Space-mean speed of target over the segment [km/h]",
 "Leading_Speed_kmh": "Space-mean speed of leader over the segment [km/h]",
 "Speed_Difference_kmh": "Target minus leader speed [km/h]; negative = target slower (opening)",
 "Off_Centered": "Target laterally offset from leader (manual coding)",
 "Occupied": "Target carrying passenger(s) (manual coding)",
 "Roadside_Parking": "Roadside parking present at the section (manual coding)",
 "Road_Width_m": "Carriageway width [m]",
 "Parking_Width_m": "Width occupied by roadside parking at event time [m]",
 "Effective_Width_m": "Road_Width_m - Parking_Width_m [m]",
 "Flow_pcu_h": "Rolling flow, centred 30-s window at floor(TT2), from YOLO crossings [pcu/h]",
 "Flow_veh_h": "Same window, vehicle count [veh/h]",
 "Flow_pcu_h_m": "Flow_pcu_h / Effective_Width_m [pcu/h/m]",
 "Flow_Window_Eff_s": "Effective window width after truncation at recording edges [s]",
 "Flow_Edge_Window": "True if effective window < 24 s",
 "Share_3W": "PCU share of YOLO class 3W (PR, BTW, van and other non-CNG three-wheelers) in the window",
 "Share_CNG": "PCU share of CNG auto-rickshaws",
 "Share_3W_All": "Share_3W + Share_CNG",
 "Share_4W": "PCU share of 4W (cars, utility, etc.)",
 "Share_Bus": "PCU share of buses",
 "Share_Bike": "PCU share of motorcycles",
 "Share_Bicycle": "PCU share of bicycles",
 "Share_2W": "Share_Bike + Share_Bicycle",
 "QA_Frame_GE_FPS": "A frame field >= 30 in source (retained; valid entry)",
 "QA_Target_Ahead_At_L1": "TT1 < LT1: target reached line 1 before leader (dropped in data02)",
 "QA_Shared_Leader": "Same leader (LT1, LT2) used by more than one target; normal in non-lane traffic, informational",
}
dictionary = pd.DataFrame({"Variable": COLS, "Description": [DESC[c] for c in COLS]})
params = pd.DataFrame({"Parameter": ["FPS", "WINDOW_SEC", "EDGE_FRACTION", "FLOW_TIME_RULE", "PCU",
                                     "TARGET_MAP", "LEADER_MAP", "DROP_TARGET_AHEAD_AT_L1", "built_at"],
                       "Value": [FPS, WINDOW_SEC, EDGE_FRACTION, FLOW_TIME_RULE, json.dumps(PCU),
                                 json.dumps(TARGET_MAP), json.dumps(LEADER_MAP), DROP_TARGET_AHEAD_AT_L1,
                                 datetime.datetime.now().isoformat(timespec="seconds")]})

### 9. Export

In [ ]:
def write(path, main, name, extra):
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        main.to_excel(xw, sheet_name=name, index=False)
        for sh, df in extra.items():
            df.to_excel(xw, sheet_name=sh, index=False)
        for ws in xw.book.worksheets:
            ws.freeze_panes = "B2"
            for col in ws.columns:
                width = max(len(str(c.value)) if c.value is not None else 0 for c in list(col)[:50])
                ws.column_dimensions[col[0].column_letter].width = min(max(10, width + 2), 60)
    cambria_xlsx(path)
    print("wrote", path.relative_to(ROOT))

counts = lambda df: (df.groupby(["Site", "Session", "Target_Class"]).size().unstack(fill_value=0)
                       .assign(Total=lambda x: x.sum(axis=1)).reset_index())
write(OUT / "data01.xlsx", data01, "data01",
      {"dictionary": dictionary, "params": params, "flow_reproduction": rep,
       "qa_summary": qa.rename("n_flagged").reset_index().rename(columns={"index": "flag"}),
       "counts": counts(data01), "inputs": fingerprints})
write(OUT / "data02.xlsx", data02, "data02",
      {"dictionary": dictionary[dictionary.Variable != "QA_Target_Ahead_At_L1"], "params": params,
       "dropped_from_data01": dropped, "counts": counts(data02),
       "pair_counts": data02.groupby(["Target_Class", "Leader_Class"]).size().unstack(fill_value=0).reset_index()})

### 9b. data03: main analysis variables, single sheet, before pooling

The class labels stay **as coded**. Pooling (Mod_BTW into BTW, and 11 leader labels into 5 classes) is tested and applied in the pooling section of this notebook, which produces data04.

In [ ]:
DATA03_COLS = ["Obs_ID", "Site", "Session", "V_Target_Raw", "V_Leading_Raw",
               "Time_Headway_s", "Target_Speed_kmh", "Leading_Speed_kmh", "Speed_Difference_kmh",
               "Off_Centered", "Occupied", "Roadside_Parking", "Effective_Width_m",
               "Flow_pcu_h", "Flow_pcu_h_m",
               "Share_3W", "Share_CNG", "Share_4W", "Share_Bus", "Share_Bike", "Share_Bicycle"]
data03 = data02[DATA03_COLS].rename(columns={"V_Target_Raw": "V_Target", "V_Leading_Raw": "V_Leading"})
data03.to_excel(OUT / "data03.xlsx", sheet_name="data03", index=False)
cambria_xlsx(OUT / "data03.xlsx")
print("wrote data/data03.xlsx", data03.shape)

### 10. Summary

In [ ]:
print(counts(data02).to_string(index=False))
print()
print(data02.groupby("Target_Class")[["Time_Headway_s", "Target_Speed_kmh", "Leading_Speed_kmh", "Flow_pcu_h_m"]]
      .agg(["count", "mean", "std", "min", "max"]).round(3).T.to_string())

## Part B — Class pooling: data03 → data04

This notebook tests and applies the two pooling steps:

- **A. Target subtypes:** whether Mod_BTW can be pooled with BTW into one BTW class. PR is kept separate.
- **B. Leader labels:** whether the 11 coded leader labels can be collapsed into five engineering classes. The scheme is defined *a priori* from vehicle size, propulsion and wheel count, and the tests here are supporting evidence for it.

- Input: `data/data03.xlsx`.
- Outputs: `public/data/data04.xlsx` (the public analysis dataset, single sheet), plus `outputs/pooling/Pooling_Tables.xlsx` and two appendix figures in `outputs/figures/` (PNG, 600 dpi).

In [ ]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path.cwd()))
from style import PAL, use_style
from pathlib import Path
import json, datetime, warnings
import numpy as np
import pandas as pd
from scipy import stats
import pingouin as pg
import matplotlib.pyplot as plt

from openpyxl import load_workbook
from openpyxl.styles import Font
FONT_NAME = "Cambria"            # house font for all figures and Excel outputs
def cambria_xlsx(path):
    # set every cell of every sheet in an .xlsx to Cambria (header row bold)
    wb = load_workbook(path)
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for c in row:
                c.font = Font(name=FONT_NAME, bold=(c.row == 1), size=11)
    wb.save(path)
warnings.filterwarnings("ignore", category=UserWarning)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTP = ROOT / "outputs" / "tables"; OUTP.mkdir(parents=True, exist_ok=True)
OUTA = ROOT / "outputs" / "figures"; OUTA.mkdir(parents=True, exist_ok=True)   # pooling figures (appendix)
PUB = ROOT / "outputs" / "data"; PUB.mkdir(parents=True, exist_ok=True)
d = pd.read_excel(ROOT / "outputs" / "data" / "data03.xlsx", sheet_name="data03")
print(d.shape)

### PARAMS

In [ ]:
SEED = 20261106
N_BOOT = 2000                 # bootstrap replicates for the Cliff's delta CI
ALPHA = 0.05
TOST_BOUND = {"Time_Headway_s": 0.30,      # s; equivalence margin for headway (same as the CSTP pipeline)
              "Target_Speed_kmh": 2.0}     # km/h; equivalence margin for target speed (new, to be confirmed)
NEGLIGIBLE_DELTA = 0.147      # |Cliff's delta| < 0.147 = negligible (Romano et al. 2006)
TARGET_POOL = {"BTW": "BTW", "Mod_BTW": "BTW", "PR": "PR"}
LEADER_POOL = {"Car": "4W", "SUV": "4W", "Utility_V": "4W", "Man_Hauler": "4W",
               "BTW": "BTW", "Mod_BTW": "BTW", "CNG": "CNG",   # CNG kept separate (Tahsin, 2026-11-06)
               "PR": "NMT", "Van": "NMT", "Bicycle": "NMT",   # non-motorised merged (Tahsin, 2026-11-06)
               "Bike": "MT_2W"}
LEADER_ORDER = ["4W", "BTW", "CNG", "MT_2W", "NMT"]
# Pooling rule for target subtypes (A): pool if headway equivalence holds (TOST p < ALPHA)
# AND |delta| is negligible for both headway and target speed.

### Helper statistics

In [ ]:
rng = np.random.default_rng(SEED)

def cliffs_delta(x, y):
    x, y = np.asarray(x), np.asarray(y)
    gt = (x[:, None] > y[None, :]).sum(); lt = (x[:, None] < y[None, :]).sum()
    return (gt - lt) / (len(x) * len(y))

def delta_ci(x, y, B=N_BOOT):
    x, y = np.asarray(x), np.asarray(y)
    bs = [cliffs_delta(rng.choice(x, len(x)), rng.choice(y, len(y))) for _ in range(B)]
    return np.percentile(bs, [2.5, 97.5])

def magnitude(dl):
    a = abs(dl)
    return "negligible" if a < 0.147 else "small" if a < 0.33 else "medium" if a < 0.474 else "large"

def two_group(x, y, label_x, label_y, var, bound=None):
    x, y = np.asarray(x, float), np.asarray(y, float)
    welch = stats.ttest_ind(x, y, equal_var=False)
    mw = stats.mannwhitneyu(x, y, alternative="two-sided")
    ks = stats.ks_2samp(x, y)
    ad = stats.anderson_ksamp([x, y], method=stats.PermutationMethod(n_resamples=2000, random_state=SEED))
    dl = cliffs_delta(x, y); lo, hi = delta_ci(x, y)
    bf10 = float(pg.ttest(x, y, correction=True)["BF10"].iloc[0])
    row = {"variable": var, "group_1": label_x, "group_2": label_y, "n_1": len(x), "n_2": len(y),
           "mean_1": x.mean(), "mean_2": y.mean(), "median_1": np.median(x), "median_2": np.median(y),
           "welch_t": welch.statistic, "welch_p": welch.pvalue,
           "MW_U": mw.statistic, "MW_p": mw.pvalue,
           "KS_D": ks.statistic, "KS_p": ks.pvalue,
           "AD_stat": ad.statistic, "AD_p": ad.pvalue,
           "cliffs_delta": dl, "delta_CI_lo": lo, "delta_CI_hi": hi, "delta_magnitude": magnitude(dl),
           "BF10": bf10, "BF01": 1 / bf10}
    if bound is not None:
        t = pg.tost(x, y, bound=bound)
        row.update({"TOST_bound": bound, "TOST_p": float(t["pval"].iloc[0])})
    return row

### A. Target subtypes: BTW vs Mod_BTW

Two outcomes are tested: time headway, which is the modelled outcome, and target speed. Each is tested on all data and within each site.
Distributional tests (KS, k-sample AD) and effect sizes (Cliff's δ with a bootstrap CI) describe the difference.
**TOST carries the pooling decision**, because failing to reject "no difference" is not evidence of equivalence. The JZS Bayes factor BF01 is reported as a supporting measure.

In [ ]:
rowsA = []
for scope, g in [("All", d)] + list(d.groupby("Site")):
    for var in ["Time_Headway_s", "Target_Speed_kmh"]:
        x = g.loc[g.V_Target == "BTW", var]; y = g.loc[g.V_Target == "Mod_BTW", var]
        rowsA.append({"scope": scope, **two_group(x, y, "BTW", "Mod_BTW", var, TOST_BOUND[var])})
A = pd.DataFrame(rowsA)

# Reference contrast: PR vs pooled BTW
rowsR = []
for scope, g in [("All", d)] + list(d.groupby("Site")):
    for var in ["Time_Headway_s", "Target_Speed_kmh"]:
        x = g.loc[g.V_Target == "PR", var]; y = g.loc[g.V_Target.isin(["BTW", "Mod_BTW"]), var]
        rowsR.append({"scope": scope, **two_group(x, y, "PR", "BTW+Mod_BTW", var)})
R = pd.DataFrame(rowsR)

show = ["scope", "variable", "n_1", "n_2", "mean_1", "mean_2", "welch_p", "MW_p", "KS_D", "KS_p", "AD_p",
        "cliffs_delta", "delta_CI_lo", "delta_CI_hi", "TOST_p", "BF01"]
print(A[show].round(4).to_string(index=False))
print()
print(R[[c for c in show if c != "TOST_p"]].round(4).to_string(index=False))

In [ ]:
a_all = A[A.scope == "All"].set_index("variable")
pool_target = bool((a_all.loc["Time_Headway_s", "TOST_p"] < ALPHA)
                   and (a_all["cliffs_delta"].abs() < NEGLIGIBLE_DELTA).all())
print("Pool Mod_BTW into BTW:", pool_target)

### B. Leader classes

**B1. Within-class homogeneity:** for each pooled class built from two or more raw labels, the raw labels are compared on leader speed and on the target's headway, using Kruskal–Wallis, then Holm-adjusted pairwise Mann–Whitney tests with Cliff's δ.

**B2. Between-class separation:** the five classes are compared on leader speed using Kruskal–Wallis (ε²), then pairwise Mann–Whitney (Holm), Cliff's δ and KS. For pairs that are not significantly different, a TOST is also run on leader speed with a ±2 km/h margin.

In [ ]:
d["Leader_Class"] = d["V_Leading"].map(LEADER_POOL)
assert d["Leader_Class"].notna().all()

def kw(groups):
    h, p = stats.kruskal(*groups)
    n = sum(len(g) for g in groups); k = len(groups)
    return h, p, (h - k + 1) / (n - k)          # epsilon-squared (Tomczak & Tomczak 2014)

def pairwise(df, group_col, var, order):
    rows = []
    for i, a in enumerate(order):
        for b in order[i + 1:]:
            x = df.loc[df[group_col] == a, var]; y = df.loc[df[group_col] == b, var]
            mw = stats.mannwhitneyu(x, y, alternative="two-sided"); ks = stats.ks_2samp(x, y)
            dl = cliffs_delta(x, y)
            rows.append({"group_1": a, "group_2": b, "n_1": len(x), "n_2": len(y),
                         "median_1": x.median(), "median_2": y.median(), "MW_p": mw.pvalue,
                         "KS_D": ks.statistic, "KS_p": ks.pvalue,
                         "cliffs_delta": dl, "delta_magnitude": magnitude(dl)})
    out = pd.DataFrame(rows)
    out["MW_p_holm"] = pg.multicomp(out["MW_p"].values, method="holm")[1]
    return out

# B1 within-class homogeneity
B1_omni, B1_pair = [], []
for cls in LEADER_ORDER:
    labs = sorted(k for k, v in LEADER_POOL.items() if v == cls)
    if len(labs) < 2:
        continue
    g = d[d.Leader_Class == cls]
    for var in ["Leading_Speed_kmh", "Time_Headway_s"]:
        h, p, e2 = kw([g.loc[g.V_Leading == l, var] for l in labs])
        B1_omni.append({"class": cls, "labels": ", ".join(labs), "n": len(g), "variable": var,
                        "KW_H": h, "KW_p": p, "epsilon2": e2})
        B1_pair.append(pairwise(g, "V_Leading", var, labs).assign(**{"class": cls, "variable": var}))
B1_omni = pd.DataFrame(B1_omni); B1_pair = pd.concat(B1_pair, ignore_index=True)
print(B1_omni.round(4).to_string(index=False))
print()
print(B1_pair[B1_pair.MW_p_holm < ALPHA][["class", "variable", "group_1", "group_2", "median_1", "median_2",
                                           "MW_p_holm", "cliffs_delta", "delta_magnitude"]].round(4).to_string(index=False))

In [ ]:
# B2 between-class separation on leader speed (and headway for reference)
B2_omni, B2_pair = [], []
for var in ["Leading_Speed_kmh", "Time_Headway_s"]:
    h, p, e2 = kw([d.loc[d.Leader_Class == c, var] for c in LEADER_ORDER])
    B2_omni.append({"variable": var, "KW_H": h, "KW_p": p, "epsilon2": e2, "n": len(d)})
    B2_pair.append(pairwise(d, "Leader_Class", var, LEADER_ORDER).assign(variable=var))
B2_omni = pd.DataFrame(B2_omni); B2_pair = pd.concat(B2_pair, ignore_index=True)

LEADER_TOST_BOUND = 2.0   # km/h
ns = B2_pair[(B2_pair.variable == "Leading_Speed_kmh") & (B2_pair.MW_p_holm >= ALPHA)]
tost_rows = []
for _, r in ns.iterrows():
    x = d.loc[d.Leader_Class == r.group_1, "Leading_Speed_kmh"]; y = d.loc[d.Leader_Class == r.group_2, "Leading_Speed_kmh"]
    t = pg.tost(x, y, bound=LEADER_TOST_BOUND); bf10 = float(pg.ttest(x, y)["BF10"].iloc[0])
    tost_rows.append({"group_1": r.group_1, "group_2": r.group_2, "TOST_bound_kmh": LEADER_TOST_BOUND,
                      "TOST_p": float(t["pval"].iloc[0]), "BF01": 1 / bf10})
B2_tost = pd.DataFrame(tost_rows)
print(B2_omni.round(4).to_string(index=False)); print()
print(B2_pair[B2_pair.variable == "Leading_Speed_kmh"].drop(columns="variable").round(4).to_string(index=False)); print()
print(B2_tost.round(4).to_string(index=False))

### Appendix figures (saved to outputs/figures)

In [ ]:
plt.rcParams.update({"font.family": FONT_NAME, "mathtext.fontset": "custom", "mathtext.rm": FONT_NAME,
                     "mathtext.it": FONT_NAME + ":italic", "mathtext.bf": FONT_NAME + ":bold", "font.size": 9})
fig, ax = plt.subplots(1, 2, figsize=(7.2, 2.9))
for lab, c, ls in [("PR", PAL["target_sub"]["PR"], "-"), ("BTW", PAL["target_sub"]["BTW"], "-"), ("Mod_BTW", PAL["target_sub"]["Mod_BTW"], "--")]:
    for j, var in enumerate(["Time_Headway_s", "Target_Speed_kmh"]):
        x = np.sort(d.loc[d.V_Target == lab, var]); ax[j].step(x, np.arange(1, len(x) + 1) / len(x), where="post",
                                                                 color=c, ls=ls, lw=1.3, label=f"{lab} (n={len(x)})")
ax[0].set(xlabel="Time headway (s)", ylabel="ECDF"); ax[1].set(xlabel="Target speed (km/h)")
ax[0].legend(frameon=False, fontsize=8)
for a in ax: a.grid(alpha=.3)
fig.tight_layout(); (OUTA / "target_subtypes_ECDF.png").unlink(missing_ok=True); fig.savefig(OUTA / "target_subtypes_ECDF.png", dpi=600)

fig, ax = plt.subplots(figsize=(7.2, 3.0))
labs = [l for c in LEADER_ORDER for l in sorted(k for k, v in LEADER_POOL.items() if v == c)]
data_b = [d.loc[d.V_Leading == l, "Leading_Speed_kmh"] for l in labs]
bp = ax.boxplot(data_b, tick_labels=labs, showfliers=False, patch_artist=True)
cmap = {c: PAL["leader"][c] for c in LEADER_ORDER}
for patch, l in zip(bp["boxes"], labs):
    patch.set_facecolor(cmap[LEADER_POOL[l]]); patch.set_alpha(.6)
ax.set(ylabel="Leader speed (km/h)"); ax.tick_params(axis="x", rotation=40); ax.grid(axis="y", alpha=.3)
handles = [plt.Rectangle((0, 0), 1, 1, color=cmap[c], alpha=.6) for c in LEADER_ORDER]
ax.legend(handles, LEADER_ORDER, ncol=5, frameon=False, fontsize=8, loc="lower center", bbox_to_anchor=(0.5, 1.0))
fig.tight_layout(); (OUTA / "leader_speed_by_label.png").unlink(missing_ok=True); fig.savefig(OUTA / "leader_speed_by_label.png", dpi=600)

# Time headway by leader label, separately for BTW and PR followers (target class strongly shifts headway)
fig, axes = plt.subplots(2, 1, figsize=(7.2, 5.4), sharex=True)
for ax, tgt, tag in zip(axes, ["BTW", "PR"], "ab"):
    sub = d[d.V_Target.map(TARGET_POOL) == tgt]
    data_h = [sub.loc[sub.V_Leading == l, "Time_Headway_s"] for l in labs]
    bp = ax.boxplot(data_h, tick_labels=labs, showfliers=False, patch_artist=True)
    for patch, l in zip(bp["boxes"], labs):
        patch.set_facecolor(cmap[LEADER_POOL[l]]); patch.set_alpha(.6)
    for k, x in enumerate(data_h, start=1):
        ax.text(k, 5.12, f"n={len(x)}", ha="center", va="bottom", fontsize=6.5, color="0.25")
    ax.set(ylabel="Time headway (s)", ylim=(0.4, 5.5))
    ax.set_title(f"({tag}) {tgt} followers (n={len(sub)})", loc="left", fontsize=8.5)
    ax.grid(axis="y", alpha=.3)
axes[-1].tick_params(axis="x", rotation=40)
axes[0].legend(handles, LEADER_ORDER, ncol=5, frameon=False, fontsize=8, loc="lower center", bbox_to_anchor=(0.5, 1.08))
fig.tight_layout(); (OUTA / "headway_by_leader_label.png").unlink(missing_ok=True); fig.savefig(OUTA / "headway_by_leader_label.png", dpi=600)
plt.show()

### Export the pooling tables and data04

In [ ]:
params = pd.DataFrame({"Parameter": ["SEED", "N_BOOT", "ALPHA", "TOST_BOUND", "NEGLIGIBLE_DELTA", "LEADER_TOST_BOUND",
                                     "TARGET_POOL", "LEADER_POOL", "pool_target_decision", "built_at"],
                       "Value": [SEED, N_BOOT, ALPHA, json.dumps(TOST_BOUND), NEGLIGIBLE_DELTA, LEADER_TOST_BOUND,
                                 json.dumps(TARGET_POOL), json.dumps(LEADER_POOL), pool_target,
                                 datetime.datetime.now().isoformat(timespec="seconds")]})
with pd.ExcelWriter(OUTP / "Pooling_Tables.xlsx") as xw:
    A.to_excel(xw, sheet_name="A_BTW_vs_ModBTW", index=False)
    R.to_excel(xw, sheet_name="A_ref_PR_vs_BTW", index=False)
    B1_omni.to_excel(xw, sheet_name="B1_within_omnibus", index=False)
    B1_pair.to_excel(xw, sheet_name="B1_within_pairwise", index=False)
    B2_omni.to_excel(xw, sheet_name="B2_between_omnibus", index=False)
    B2_pair.to_excel(xw, sheet_name="B2_between_pairwise", index=False)
    B2_tost.to_excel(xw, sheet_name="B2_equivalence", index=False)
    params.to_excel(xw, sheet_name="params", index=False)
cambria_xlsx(OUTP / "Pooling_Tables.xlsx")

assert pool_target, "Target pooling not supported; data04 not written (revisit decision)"
d04 = d.rename(columns={"V_Target": "Target_Type", "V_Leading": "Leader_Type"})
d04.insert(d04.columns.get_loc("Target_Type") + 1, "Target_Class", d04["Target_Type"].map(TARGET_POOL))
d04 = d04.drop(columns="Leader_Class")
d04.insert(d04.columns.get_loc("Leader_Type") + 1, "Leader_Class", d04["Leader_Type"].map(LEADER_POOL))
d04.insert(d04.columns.get_loc("Leader_Class") + 1, "Pair", d04["Target_Class"] + "_following_" + d04["Leader_Class"])
DROP_FROM_D04 = ["Roadside_Parking", "Effective_Width_m", "Flow_pcu_h", "Flow_pcu_h_m",
                 "Share_3W", "Share_CNG", "Share_4W", "Share_Bus", "Share_Bike", "Share_Bicycle"]  # not used further (Tahsin)
d04 = d04.drop(columns=DROP_FROM_D04)
d04.to_excel(PUB / "data04.xlsx", sheet_name="data04", index=False)
cambria_xlsx(PUB / "data04.xlsx")
print("data04:", d04.shape); print(list(d04.columns))
print(d04.groupby(["Target_Class", "Leader_Class"]).size().unstack())

## Part C — Appendix figures (named, not numbered)

| Figure | Content | Sources |
|---|---|---|
| traffic_state_rolling_flow | Traffic state during each survey session: rolling 30-s flow stacked by YOLO class, composition shares, and the sampled events | YOLO crossing logs and data02 |
| yolo_detector_performance | YOLO detector performance on the validation split: normalised confusion matrix, PR curves and training history | model `best.pt`, labelled data, `results.csv` |
| event_timing | Timing of the sampled events within each session, by target class | data02 |

The pooling figures (target_subtypes_ECDF, leader_speed_by_label) are produced by the pooling section of this notebook into the same folder. All outputs are PNG at 600 dpi in Cambria, written to `outputs/figures/`.

In [ ]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path.cwd()))
from style import PAL, use_style
from pathlib import Path
import json, shutil, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from openpyxl import load_workbook
from openpyxl.styles import Font

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROJ = ROOT
OUTA = ROOT / "outputs" / "figures"; OUTA.mkdir(parents=True, exist_ok=True)
OUTT = ROOT / "outputs" / "tables"; OUTT.mkdir(parents=True, exist_ok=True)
RAW = ROOT / "data" / "raw"
YOLO_DIR = RAW / "yolo"            # training history and cached validation results of the YOLO26n detector

FONT_NAME = "Cambria"
plt.rcParams.update({"font.family": FONT_NAME, "mathtext.fontset": "custom", "mathtext.rm": FONT_NAME,
                     "mathtext.it": FONT_NAME + ":italic", "mathtext.bf": FONT_NAME + ":bold", "font.size": 9})
def save(fig, name):
    p = OUTA / f"{name}.png"; p.unlink(missing_ok=True)
    fig.savefig(p, dpi=600, bbox_inches="tight"); print("saved", p.relative_to(ROOT))
def cambria_xlsx(path):
    wb = load_workbook(path)
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for c in row:
                c.font = Font(name=FONT_NAME, bold=(c.row == 1), size=11)
    wb.save(path)

### PARAMS

In [ ]:
WINDOW_SEC = 30.0; EDGE_FRACTION = 0.8          # identical to 01_build_data
PCU = {"3W": 2.00, "4W": 1.00, "Bicycle": 0.50, "Bike": 0.75, "Bus": 3.00, "CNG": 0.75}
CLASS_ORDER = ["3W", "CNG", "4W", "Bus", "Bike", "Bicycle"]
CLASS_COL = PAL["yolo_class"]
TARGET_COL = PAL["target"]
SESSIONS = {"SP": {"log": "SP_log_M.xlsx", "label": "Shahjahanpur, session 1 (SP)", "video": "SP.mp4"},
            "SP2": {"log": "SP2_log.xlsx", "label": "Shahjahanpur, session 2 (SP2)", "video": "SP2 (6-7-26).mp4"},
            "Tikatuli": {"log": "Tikatuli Merged.xlsx", "label": "Tikatuli", "video": "Tikatuli 1.mp4"}}

# Camera views: YOLO count lines (from the tracking code; confirmed by the logged crossing centroids,
# which all sit just past these y-values: SP cy>=601, SP2 cy>=701, Tikatuli cy>=901)
COUNT_LINE = {"SP": ((100, 600), (1800, 600)), "SP2": ((200, 700), (1800, 700)), "Tikatuli": ((400, 900), (2200, 900))}
FRAME_TIME = {"SP": "12:45", "SP2": "13:35", "Tikatuli": "15:57"}   # mm:ss in each video file (Tahsin); Tikatuli from "Tikatuli 1.mp4"

# YOLO evaluation
MODEL_PT = YOLO_DIR / "best.pt"   # optional: weights from the YOLO repository, only needed to re-evaluate
LABELLED = YOLO_DIR / "Training Data (labelled)"   # optional: labelled images, only needed to re-evaluate
YOLO_EVAL_CACHE = YOLO_DIR / "yolo_eval_cache.json"
RUN_YOLO_EVAL = not YOLO_EVAL_CACHE.exists()    # re-evaluate only when no cache exists (delete the json to force)
EVAL_KW = dict(imgsz=1280, batch=4, conf=0.001, iou=0.7, plots=True, verbose=False, device=0)
CM_CONF = 0.26               # detection confidence used in tracking; the confusion matrix is computed at this threshold

### Shared data

In [ ]:
d02 = pd.read_excel(ROOT / "outputs" / "data" / "data02.xlsx", sheet_name="data02")

def per_second_flow(log):
    log = log.sort_values("time_sec")
    tt = log.time_sec.to_numpy(float); cl = log.class_name.to_numpy()
    t0, t1 = tt.min(), tt.max()
    secs = np.arange(np.floor(t0), np.ceil(t1) + 1)
    lo = np.clip(secs - WINDOW_SEC / 2, t0, None); hi = np.clip(secs + WINDOW_SEC / 2, None, t1); eff = hi - lo
    il = np.searchsorted(tt, lo, "left"); ih = np.searchsorted(tt, hi, "right")
    out = pd.DataFrame({"t_sec": secs, "eff": eff, "edge": eff < EDGE_FRACTION * WINDOW_SEC})
    for k in CLASS_ORDER:
        ck = np.r_[0, np.cumsum(np.where(cl == k, PCU[k], 0.0))]
        out[k] = (ck[ih] - ck[il]) * 3600 / eff
    out["total"] = out[CLASS_ORDER].sum(axis=1)
    return out

flows = {s: per_second_flow(pd.read_excel(RAW / "yolo_crossings" / c["log"])) for s, c in SESSIONS.items()}
summary = []
for s, f in flows.items():
    core = f[~f.edge]
    ev = d02[d02.Session == s]
    summary.append({"Session": s, "duration_min": (f.t_sec.max() - f.t_sec.min()) / 60,
                    "flow_mean_pcu_h": core.total.mean(), "flow_sd": core.total.std(),
                    "flow_p05": core.total.quantile(.05), "flow_p95": core.total.quantile(.95),
                    "flow_max": core.total.max(), "n_events": len(ev),
                    "first_event_s": ev.TT2.min(), "last_event_s": ev.TT2.max(),
                    **{f"mean_share_{k}": (core[k] / core.total).mean() for k in CLASS_ORDER}})
summary = pd.DataFrame(summary)
summary.round(3)

### Traffic state during the survey sessions

In [ ]:
durs = np.array([flows[s].t_sec.max() - flows[s].t_sec.min() for s in SESSIONS])
fig = plt.figure(figsize=(7.2, 8.6))
outer = gridspec.GridSpec(3, 1, hspace=0.42, figure=fig)
for i, (s, cfg) in enumerate(SESSIONS.items()):
    f = flows[s]; m = f.t_sec / 60
    inner = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=outer[i], height_ratios=[2.2, 1], hspace=0.08)
    a = fig.add_subplot(inner[0]); b = fig.add_subplot(inner[1], sharex=a)
    a.stackplot(m, *[f[k] for k in CLASS_ORDER], colors=[CLASS_COL[k] for k in CLASS_ORDER], alpha=.85, lw=0)
    a.plot(m, f.total, color="k", lw=.6)
    for side in [f[f.edge & (f.t_sec < f.t_sec.median())], f[f.edge & (f.t_sec > f.t_sec.median())]]:
        if len(side): a.axvspan(side.t_sec.min() / 60, side.t_sec.max() / 60, color="0.85", zorder=0, lw=0)
    ymax = f.total.max() * 1.12
    ev = d02[d02.Session == s]
    for tc, yoff in [("BTW", 1.0), ("PR", 0.955)]:
        e = ev[ev.Target_Class == tc]
        a.plot(e.TT2 / 60, np.full(len(e), ymax * yoff), "|", ms=4.5, mew=.6, color=TARGET_COL[tc])
    a.set_ylim(0, ymax * 1.03); a.set_ylabel("Flow (pcu/h)")
    a.set_title(f"({'abc'[i]}) {cfg['label']}: {len(ev)} sampled events", loc="left", fontsize=9)
    shares = f[CLASS_ORDER].div(f.total.replace(0, np.nan), axis=0).fillna(0)
    b.stackplot(m, *[shares[k] for k in CLASS_ORDER], colors=[CLASS_COL[k] for k in CLASS_ORDER], alpha=.85, lw=0)
    b.set_ylim(0, 1); b.set_ylabel("PCU share"); b.set_xlim(m.min(), m.max())
    plt.setp(a.get_xticklabels(), visible=False)
    if i == 2: b.set_xlabel("Time from start of session (min)")
    for ax in (a, b): ax.grid(alpha=.25, lw=.5)
handles = [plt.Rectangle((0, 0), 1, 1, color=CLASS_COL[k], alpha=.85) for k in CLASS_ORDER] + \
          [Line2D([], [], color="k", lw=.8), Line2D([], [], marker="|", ls="", color=TARGET_COL["BTW"], ms=7),
           Line2D([], [], marker="|", ls="", color=TARGET_COL["PR"], ms=7), plt.Rectangle((0, 0), 1, 1, color="0.85")]
labels = CLASS_ORDER + ["Total", "Sampled event: BTW", "Sampled event: PR", "Edge (window < 24 s)"]
fig.legend(handles, labels, ncol=5, frameon=False, fontsize=8, loc="lower center", bbox_to_anchor=(0.5, 0.0))
fig.subplots_adjust(bottom=0.09, top=0.97, left=0.1, right=0.98)
save(fig, "traffic_state_rolling_flow"); plt.show()

### Site views: one still image per session at the chosen time (no overlays)

### YOLO detector performance

The trained model is evaluated on the **validation split** (287 images, 3,700 instances).
The `Test` folder is a byte-identical copy of `Val`, so there is no independent test set. Because `best.pt` was selected on Val, these figures are validation metrics, not held-out performance.
Neither split shares any image with Train.
The results are cached in `yolo_eval_cache.json`, so later runs only redraw the figure. Delete the json file to re-evaluate.

In [ ]:
def evaluate():
    from ultralytics import YOLO
    import yaml, tempfile
    cfg = {"path": str(LABELLED).replace("\\", "/"), "train": "Train/images", "val": "Val/images", "test": "Test/images",
           "nc": 6, "names": {0: "Bicycle", 1: "Bike", 2: "3W", 3: "CNG", 4: "4W", 5: "Bus"}}
    tmp = Path(tempfile.mkdtemp()); yml = tmp / "data_local.yaml"; yml.write_text(yaml.safe_dump(cfg))
    model = YOLO(str(MODEL_PT)); res = {}
    for split in ["val"]:
        m = model.val(data=str(yml), split=split, project=str(tmp), name=split, **EVAL_KW)
        names = [m.names[i] for i in range(len(m.names))]
        curves = {r[3] + "_vs_" + r[2]: {"x": np.asarray(r[0]).tolist(), "y": np.asarray(r[1]).tolist()}
                  for r in m.box.curves_results}
        # confusion matrix at the tracking confidence threshold
        mc = model.val(data=str(yml), split=split, project=str(tmp), name=split + "_cm",
                       **{**EVAL_KW, "conf": CM_CONF})
        res[split] = {"names": names, "mAP50": float(m.box.map50), "mAP50_95": float(m.box.map),
                      "precision": float(m.box.mp), "recall": float(m.box.mr),
                      "per_class_AP50": dict(zip(names, map(float, m.box.ap50))),
                      "per_class_AP50_95": dict(zip(names, map(float, m.box.ap))),
                      "confusion_matrix": mc.confusion_matrix.matrix.tolist(), "curves": curves}
    shutil.rmtree(tmp, ignore_errors=True)
    res["meta"] = {"model": str(MODEL_PT.name), "eval_kw": EVAL_KW, "cm_conf": CM_CONF,
                   "evaluated_at": datetime.datetime.now().isoformat(timespec="seconds")}
    YOLO_EVAL_CACHE.write_text(json.dumps(res))
    return res

ev = evaluate() if RUN_YOLO_EVAL else json.loads(YOLO_EVAL_CACHE.read_text())
for split in ["val"]:
    r = ev[split]
    print(f"{split}: mAP50={r['mAP50']:.3f} mAP50-95={r['mAP50_95']:.3f} P={r['precision']:.3f} R={r['recall']:.3f}")
    print("   AP50 by class:", {k: round(v, 3) for k, v in r["per_class_AP50"].items()})
print(list(ev["val"]["curves"]))

In [ ]:
r = ev["val"]; names = r["names"]
cm = np.array(r["confusion_matrix"])            # ultralytics layout: rows = predicted, cols = true; last = background
labels_cm = names + ["Background"]
cm_true = cm.T                                   # rows = true class
row_sum = cm_true.sum(axis=1, keepdims=True)
cmn = np.divide(cm_true, row_sum, out=np.zeros_like(cm_true, dtype=float), where=row_sum > 0)

fig = plt.figure(figsize=(7.2, 6.0))
gs = gridspec.GridSpec(2, 2, height_ratios=[1.15, 1], hspace=0.68, wspace=0.32, figure=fig)
a = fig.add_subplot(gs[0, 0])
show = cmn[:-1, :]                               # true background row has no meaning for recall
im = a.imshow(show, cmap="Blues", vmin=0, vmax=1)
for i in range(show.shape[0]):
    for j in range(show.shape[1]):
        if show[i, j] >= 0.005:
            a.text(j, i, f"{show[i, j]:.2f}", ha="center", va="center", fontsize=6.5,
                   color="w" if show[i, j] > .55 else "k")
a.set_xticks(range(len(labels_cm))); a.set_xticklabels(labels_cm, rotation=45, ha="right", fontsize=7)
a.set_yticks(range(len(names))); a.set_yticklabels(names, fontsize=7)
a.set_xlabel("Predicted"); a.set_ylabel("True")
a.set_title(f"(a) Confusion matrix, validation split (conf = {CM_CONF})", loc="left", fontsize=8.5)

b = fig.add_subplot(gs[0, 1])
pr_key = [k for k in r["curves"] if k.lower().startswith("precision_vs_recall")][0]
x = np.array(r["curves"][pr_key]["x"]); Y = np.array(r["curves"][pr_key]["y"])
cols = plt.cm.tab10(np.arange(len(names)))
for k, nm in enumerate(names):
    b.plot(x, Y[k], lw=1, color=cols[k], label=f"{nm} ({r['per_class_AP50'][nm]:.2f})")
b.plot(x, Y.mean(axis=0), lw=2, color="k", label=f"All classes ({r['mAP50']:.2f})")
b.set(xlabel="Recall", ylabel="Precision", xlim=(0, 1), ylim=(0, 1.02))
b.set_title("(b) Precision-recall, validation split (AP@0.5)", loc="left", fontsize=8.5)
b.legend(fontsize=6.5, frameon=False, loc="lower left"); b.grid(alpha=.25)

c = fig.add_subplot(gs[1, :])
hist = pd.read_csv(YOLO_DIR / "results.csv"); hist.columns = [s.strip() for s in hist.columns]
c.plot(hist.epoch, hist["metrics/mAP50(B)"], color=PAL["training"]["mAP50"], lw=1.3, label="mAP@0.5 (validation)")
c.plot(hist.epoch, hist["metrics/mAP50-95(B)"], color=PAL["training"]["mAP50_95"], lw=1.3, label="mAP@0.5:0.95 (validation)")
c.plot(hist.epoch, hist["metrics/precision(B)"], color="0.4", lw=.9, ls="--", label="Precision (validation)")
c.plot(hist.epoch, hist["metrics/recall(B)"], color="0.4", lw=.9, ls=":", label="Recall (validation)")
c.set(xlabel="Training epoch", ylabel="Metric", ylim=(0, 1), xlim=(1, hist.epoch.max()))
c.set_title(f"(c) Training history (best.pt on validation: mAP@0.5 = {r['mAP50']:.3f}, mAP@0.5:0.95 = {r['mAP50_95']:.3f})",
            loc="left", fontsize=8.5)
c.legend(fontsize=7, frameon=False, ncol=4, loc="lower right"); c.grid(alpha=.25)
save(fig, "yolo_detector_performance"); plt.show()

### Timing of the sampled events

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(7.2, 4.6), sharex=False)
for ax, (s, cfg) in zip(axes, SESSIONS.items()):
    f = flows[s]; ev_s = d02[d02.Session == s]
    bins = np.arange(0, f.t_sec.max() / 60 + 2, 2)       # 2-min bins
    ax.hist([ev_s.loc[ev_s.Target_Class == "BTW", "TT2"] / 60, ev_s.loc[ev_s.Target_Class == "PR", "TT2"] / 60],
            bins=bins, stacked=True, color=[TARGET_COL["BTW"], TARGET_COL["PR"]], alpha=.85,
            label=["BTW", "PR"], edgecolor="w", lw=.4)
    ax.set_xlim(0, f.t_sec.max() / 60); ax.set_ylabel("Events / 2 min")
    nb, npr = (ev_s.Target_Class == "BTW").sum(), (ev_s.Target_Class == "PR").sum()
    ax.set_title(f"({'abc'[list(SESSIONS).index(s)]}) {cfg['label']}: BTW {nb}, PR {npr}", loc="left", fontsize=8.5)
    ax.grid(alpha=.25, axis="y")
axes[0].legend(frameon=False, fontsize=8, ncol=2, loc="upper right")
axes[-1].set_xlabel("Time from start of session (min)")
fig.tight_layout()
save(fig, "event_timing"); plt.show()

### Export the appendix figure data

In [ ]:
yolo_tbl = pd.DataFrame([{"split": sp, "class": "All", "AP50": ev[sp]["mAP50"], "AP50_95": ev[sp]["mAP50_95"],
                          "precision": ev[sp]["precision"], "recall": ev[sp]["recall"]} for sp in ["val"]] +
                        [{"split": sp, "class": k, "AP50": ev[sp]["per_class_AP50"][k], "AP50_95": ev[sp]["per_class_AP50_95"][k]}
                         for sp in ["val"] for k in ev[sp]["names"]])
cm_df = pd.DataFrame(cmn, index=["True_" + l for l in labels_cm], columns=["Pred_" + l for l in labels_cm]).reset_index()
xl = OUTT / "Appendix_Figure_Data.xlsx"
with pd.ExcelWriter(xl) as xw:
    summary.to_excel(xw, sheet_name="traffic_state_summary", index=False)
    yolo_tbl.to_excel(xw, sheet_name="yolo_metrics", index=False)
    cm_df.to_excel(xw, sheet_name="yolo_confusion_val_rownorm", index=False)
    pd.DataFrame({"Parameter": ["WINDOW_SEC", "EDGE_FRACTION", "PCU", "COUNT_LINE", "FRAME_TIME",
                                "EVAL_KW", "CM_CONF", "yolo_eval_at"],
                  "Value": [WINDOW_SEC, EDGE_FRACTION, json.dumps(PCU), json.dumps(COUNT_LINE), json.dumps(FRAME_TIME), json.dumps(EVAL_KW), CM_CONF, ev["meta"]["evaluated_at"]]}
                 ).to_excel(xw, sheet_name="params", index=False)
cambria_xlsx(xl); print("wrote", xl.relative_to(ROOT))
print(summary[["Session", "duration_min", "flow_mean_pcu_h", "flow_sd", "flow_p05", "flow_p95", "flow_max", "n_events",
               "first_event_s", "last_event_s"]].round(1).to_string(index=False))